<a href="https://colab.research.google.com/github/MulkernHunter/INFO-4670/blob/main/Hunter_Mulkern_hm0940_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Name: Hunter_Mulkern_hm0940_Project2.py
# SECTION 1: Header & Grade Lookup

"""
================================================================
Course: CSCE 1030
Assignment: Project 2 - Student Grade Management System
Lab Section: CSCE 1030.
Team Members (Up to 4 total):
1. Hunter Mulkern (EUID: hm0940) - Submitter
Contribution Overview:
- Hunter Mulkern: Completed the entire project individually.
================================================================
"""


GRADE_BOUNDARIES = (90, 80, 70, 60)
GRADE_LETTERS = ("A", "B", "C", "D", "F")
GRADE_POINTS = (4.0, 3.0, 2.0, 1.0, 0.0)

In [ ]:
# SECTION 2: Name Formatting & GPA Calculation

def format_student_name(full_name):
    """
    Purpose: Remove extra spaces from a name and put it in Title Case.
    Input:   full_name (str) - the name the user typed
    Returns: str - the cleaned up name (example: "  jOHN   doe " -> "John Doe")
    """
    words = full_name.strip().split()
    clean_words = []
    for word in words:
        clean_words.append(word.capitalize())
    return " ".join(clean_words)


def calculate_gpa(scores, scale=4.0):
    """
    Purpose: Find the average, GPA, and letter grade for a list of scores.
    Input:   scores (list of floats), scale (float, default 4.0)
    Returns: tuple (avg, gpa, letter)
    """
    if len(scores) == 0:
        return 0.0, 0.0, "F"

    avg = sum(scores) / len(scores)


    position = 4
    for i in range(len(GRADE_BOUNDARIES)):
        if avg >= GRADE_BOUNDARIES[i]:
            position = i
            break

    letter = GRADE_LETTERS[position]
    gpa = GRADE_POINTS[position] * (scale / 4.0)
    return avg, gpa, letter

In [ ]:
# SECTION 3: Academic Standing & Status Filter

def evaluate_risk_standing(gpa, probation_cutoff=2.0, deans_cutoff=3.5):
    """
    Purpose: Decide a student's academic standing from their GPA.
    Input:   gpa (float), probation_cutoff (float, default 2.0),
             deans_cutoff (float, default 3.5)
    Returns: str - "Dean's List", "Good Standing", or "Probation Risk"
    """
    if gpa >= deans_cutoff:
        return "Dean's List"
    elif gpa < probation_cutoff:
        return "Probation Risk"
    else:
        return "Good Standing"


def filter_students_by_status(student_ids, student_gpas, target_status):
    """
    Purpose: Get the IDs of students who have a certain standing.
    Input:   student_ids (list), student_gpas (dict),
             target_status (str) - "All Students" or a standing name
    Returns: list - the student IDs that match
    """
    matching_ids = []
    for sid in student_ids:
        if target_status == "All Students":
            matching_ids.append(sid)
        elif evaluate_risk_standing(student_gpas[sid]) == target_status:
            matching_ids.append(sid)
    return matching_ids

In [ ]:
# SECTION 4: Class Analytics Summary

def generate_analytics_summary(student_grades):
    """
    Purpose: Combine every score in the class and find the statistics.
    Input:   student_grades (dict) - student ID -> list of scores
    Returns: dict with keys "count", "highest", "lowest", "average"
    """

    all_scores = [score for score_list in student_grades.values() for score in score_list]

    if len(all_scores) == 0:
        return {"count": 0, "highest": 0.0, "lowest": 0.0, "average": 0.0}

    summary = {}
    summary["count"] = len(all_scores)
    summary["highest"] = max(all_scores)
    summary["lowest"] = min(all_scores)
    summary["average"] = sum(all_scores) / len(all_scores)
    return summary

In [ ]:
# SECTION 5: Menu Display & Sorting Helpers

def display_menu():
    """
    Purpose: Print the main menu.
    Input:   none
    Returns: none
    """
    print("=" * 50)
    print("STUDENT GRADE MANAGEMENT SYSTEM - PROJECT 2")
    print("=" * 50)
    print("1. Input & Record Student Profile & Grades")
    print("2. Compute GPAs & Academic Summaries")
    print("3. Display Formatted Grade Roster (Sorted)")
    print("4. Search Student Profile (ID or Name)")
    print("5. Class Statistics & Score Analytics")
    print("0. Exit Portal")
    print("=" * 50)


def get_sid(record):
    """
    Purpose: Sorting key - return the student ID from a record tuple.
    Input:   record (tuple) - (sid, name, gpa, status)
    Returns: int - the student ID
    """
    return record[0]


def get_name(record):
    """
    Purpose: Sorting key - return the lowercase name from a record tuple.
    Input:   record (tuple) - (sid, name, gpa, status)
    Returns: str - the name in lowercase
    """
    return record[1].lower()


def get_gpa(record):
    """
    Purpose: Sorting key - return the GPA from a record tuple.
    Input:   record (tuple) - (sid, name, gpa, status)
    Returns: float - the GPA
    """
    return record[2]

In [ ]:
# SECTION 6: Option 1 - Add Student

def add_student(student_ids, student_names, student_grades, student_gpas, enrolled_courses):
    """
    Purpose: Option 1 - add a new student with their courses and scores.
    Input:   the five data collections (changed in place)
    Returns: none
    """
    print("--- Option 1: Input & Record Student Profile & Grades ---")


    while True:
        sid_text = input("Enter 4-digit Student ID: ").strip()
        if not sid_text.isdigit() or len(sid_text) != 4:
            print("[ERROR] Student ID must be exactly 4 digits.")
        elif int(sid_text) in student_ids:
            print("[ERROR] Student ID already exists.")
        else:
            break
    sid = int(sid_text)


    while True:
        name = format_student_name(input("Enter student full name: "))
        if name != "":
            break
        print("[ERROR] Name cannot be blank.")


    raw_courses = input("Enter course codes separated by spaces (e.g., CSCE1030 MATH1710): ")
    courses = [c.strip().upper() for c in raw_courses.split()]
    for course in courses:
        enrolled_courses.add(course)

    # Scores: check each one without try-except
    while True:
        raw_scores = input("Enter numeric scores separated by commas (e.g., 88.5, 92.0, 79.0):\n")
        pieces = raw_scores.split(",")
        scores = []
        all_valid = True
        for piece in pieces:
            piece = piece.strip()
            if piece.replace(".", "", 1).isdigit() and float(piece) <= 100:
                scores.append(float(piece))
            else:
                all_valid = False
        if all_valid:
            break
        print("[ERROR] Scores must be numbers from 0 to 100 separated by commas.")

    # Save everything
    student_ids.append(sid)
    student_names.append(name)
    student_grades[sid] = scores
    avg, gpa, letter = calculate_gpa(scores)
    student_gpas[sid] = gpa

    print(f"[SUCCESS] Record added for {name} (ID: {sid}).")

In [ ]:
# SECTION 7: Option 2 - Compute GPAs

def compute_gpas(student_ids, student_grades, student_gpas):
    """
    Purpose: Option 2 - recalculate every GPA and print a summary table.
    Input:   student_ids (list), student_grades (dict), student_gpas (dict)
    Returns: none
    """
    print("--- Option 2: Compute GPAs & Academic Summaries ---")

    if len(student_ids) == 0:
        print("[NOTICE] No student records found.")
        return

    print(f"{'ID':<6} | {'Avg Score':>9} | {'Letter':^6} | {'GPA':>4}")
    print("-" * 40)
    for sid in student_ids:
        avg, gpa, letter = calculate_gpa(student_grades[sid])
        student_gpas[sid] = gpa
        avg_text = f"{avg:.1f}%"
        print(f"{sid:<6} | {avg_text:>9} | {letter:^6} | {gpa:>4.1f}")

In [ ]:
# SECTION 8: Option 3 - Grade Roster

def display_roster(student_ids, student_names, student_gpas):
    """
    Purpose: Option 3 - show a filtered and sorted roster table.
    Input:   student_ids (list), student_names (list), student_gpas (dict)
    Returns: none
    """
    print("--- Option 3: Display Formatted Grade Roster ---")

    border = input("Enter border character (default *): ").strip()
    if border == "":
        border = "*"
    else:
        border = border[0]


    print("Academic Standing Filter")
    print("1: All Students")
    print("2: Dean's List")
    print("3: Good Standing")
    print("4: Probation Risk")
    while True:
        filter_choice = input("Select filter (1-4): ").strip()
        if filter_choice in ("1", "2", "3", "4"):
            break
        print("[ERROR] Invalid filter option. Please enter 1-4.")

    if filter_choice == "1":
        target_status = "All Students"
    elif filter_choice == "2":
        target_status = "Dean's List"
    elif filter_choice == "3":
        target_status = "Good Standing"
    else:
        target_status = "Probation Risk"

    filtered_ids = filter_students_by_status(student_ids, student_gpas, target_status)


    print("Sort Options")
    print("1: Sort by Student ID (ascending)")
    print("2: Sort by GPA (descending)")
    print("3: Sort by Name (alphabetical)")
    while True:
        sort_choice = input("Select sort option (1-3): ").strip()
        if sort_choice in ("1", "2", "3"):
            break
        print("[ERROR] Invalid sort option. Please enter 1-3.")


    records = []
    for sid in filtered_ids:
        name = student_names[student_ids.index(sid)]
        gpa = student_gpas[sid]
        status = evaluate_risk_standing(gpa)
        records.append((sid, name, gpa, status))

    if sort_choice == "1":
        records.sort(key=get_sid)
    elif sort_choice == "2":
        records.sort(key=get_gpa, reverse=True)
    else:
        records.sort(key=get_name)


    print(border * 60)
    print(f"{'GRADE ROSTER':^60}")
    print(border * 60)
    print(f"{'ID':^8}{'Name':^22}{'GPA':^8}{'Status':^22}")
    print(border * 60)
    if len(records) == 0:
        print(f"{'No students match this filter.':^60}")
    for record in records:
        sid, name, gpa, status = record
        print(f"{sid:^8}{name:^22}{gpa:^8.1f}{status:^22}")
    print(border * 60)

In [ ]:
# SECTION 9: Options 4 & 5 - Search & Class Statistics

def search_student(student_ids, student_names, student_grades):
    """
    Purpose: Option 4 - search by ID or part of a name and show a profile card.
    Input:   student_ids (list), student_names (list), student_grades (dict)
    Returns: none
    """
    print("--- Option 4: Search Student Records ---")

    query = input("Enter Student ID or Name Substring to search: ").strip()
    found = False

    if query != "":
        for i in range(len(student_ids)):
            sid = student_ids[i]
            name = student_names[i]
            if query.lower() in str(sid) or query.lower() in name.lower():
                found = True
                scores = student_grades[sid]
                avg, gpa, letter = calculate_gpa(scores)
                standing = evaluate_risk_standing(gpa)

                print("=" * 40)
                print(f"STUDENT PROFILE: {name.upper()}")
                print("=" * 40)
                print(f"Student ID   : {sid}")
                print(f"Scores       : {scores}")
                print(f"Average      : {avg:.1f}%")
                print(f"Letter Grade : {letter}")
                print(f"GPA          : {gpa:.1f}")
                print(f"Standing     : {standing}")
                if standing == "Probation Risk":
                    print("[FLAG] Academic intervention required.")
                elif standing == "Dean's List":
                    print("[EXCELLENT] Honor roll recognition.")
                print("=" * 40)

    if not found:
        print(f"[NOTICE] No student records found matching '{query}'.")


def class_statistics(student_grades, enrolled_courses):
    """
    Purpose: Option 5 - show class-wide score statistics and the course list.
    Input:   student_grades (dict), enrolled_courses (set)
    Returns: none
    """
    print("--- Option 5: Class Analytics & Score Metrics ---")

    summary = generate_analytics_summary(student_grades)
    print(f"Total Individual Scores Recorded : {summary['count']}")
    print(f"Highest Individual Score         : {summary['highest']:.1f}")
    print(f"Lowest Individual Score          : {summary['lowest']:.1f}")
    print(f"Overall Class Score Average      : {summary['average']:.1f}%")

    print("Enrolled System Courses:")
    print(", ".join(sorted(enrolled_courses)))

In [ ]:
# SECTION 10: Pre-Seeded Data & Main Menu Loop

student_ids = [1001, 1002, 1003, 1004, 1005]
student_names = ["Maria Garcia", "Kevin Brooks", "Aisha Patel", "Daniel Kim", "Olivia Turner"]
student_grades = {
    1001: [94.0, 91.5, 97.0, 89.5],
    1002: [78.0, 82.5, 74.0, 80.5],
    1003: [85.0, 88.0, 91.0, 83.0],
    1004: [62.0, 55.5, 68.0, 58.5],
    1005: [99.0, 95.5, 92.0, 96.5],
}
student_gpas = {}
for sid in student_ids:
    avg, gpa, letter = calculate_gpa(student_grades[sid])
    student_gpas[sid] = gpa
enrolled_courses = {"CSCE1030", "MATH1710", "ENGL1310"}



system_active = True
while system_active:
    display_menu()
    choice = input("Enter option (0-5): ").strip()

    if choice == "1":
        add_student(student_ids, student_names, student_grades, student_gpas, enrolled_courses)
    elif choice == "2":
        compute_gpas(student_ids, student_grades, student_gpas)
    elif choice == "3":
        display_roster(student_ids, student_names, student_gpas)
    elif choice == "4":
        search_student(student_ids, student_names, student_grades)
    elif choice == "5":
        class_statistics(student_grades, enrolled_courses)
    elif choice == "0":
        print("Logging out of Academic Management System. Goodbye!")
        system_active = False
    else:
        print("[ERROR] Invalid choice. Please enter an option between 0 and 5.")

    if choice in ("1", "2", "3", "4", "5"):
        input("\nPress Enter to return to Main Menu...")

STUDENT GRADE MANAGEMENT SYSTEM - PROJECT 2
1. Input & Record Student Profile & Grades
2. Compute GPAs & Academic Summaries
3. Display Formatted Grade Roster (Sorted)
4. Search Student Profile (ID or Name)
5. Class Statistics & Score Analytics
0. Exit Portal
--- Option 5: Class Analytics & Score Metrics ---
Total Individual Scores Recorded : 20
Highest Individual Score         : 99.0
Lowest Individual Score          : 55.5
Overall Class Score Average      : 83.0%
Enrolled System Courses:
CSCE1030, ENGL1310, MATH1710
